# T1 · J-lens 리서치 & 실행

**MindScope X · 이강훈 · 2026-09-20** — 로드맵 T1 (9/10 전동준님 배정)

| 절 | 내용 | 실행 |
|---|---|---|
| 1 | 논문 요약·리뷰 | 읽기 |
| 2 | J-lens 원리 | 읽기 |
| 3 | 우리 연구와의 연결 / 할 일 | 읽기 |
| 4 | 개념 검증 (numpy) | **CPU 즉시** |
| 5~9 | **Colab 실행: 설치 → Qwen 로드 → 렌즈 → 검증 → 함정 궤적** | **Colab GPU** |
| 10 | 한계·진행상황 | 읽기 |


---
# 1. 논문 요약·리뷰

> **Verbalizable Representations Form a Global Workspace in Language Models**
> Gurnee\*, Sofroniew\*, … Lindsey\*† (Anthropic), 2026-07-06
> [논문](https://transformer-circuits.pub/2026/workspace/index.html) · [코드](https://github.com/anthropics/jacobian-lens)

## 1.1 한 줄 요약

LLM 안에는 **"말로 꺼낼 수 있는 표현"의 특권적 집합(J-space)**이 있고, 그걸 읽는 도구가 **J-lens**다.

## 1.2 주장하는 것

- **J-space** = J-lens 벡터들의 **희소 조합**으로 표현되는 부분공간. 동시 점유 **k ≈ 25 이하**
- 이게 인지과학의 **Global Workspace Theory**와 **기능적으로** 대응한다
- 다섯 속성: 보고 가능 / 하향식 조절 / 내부 추론 / 유연한 일반화 / **선택성**

> 저자 유보: *"GWT가 뇌에 부여한 전체 아키텍처를 재현한다고 주장하지 않는다… broadcast는 순환 루프가 아니라 단일 forward pass 안에서 일어난다."*

## 1.3 ⭐ 우리에게 가장 중요한 실험 — 선택성

같은 스페인어 지문으로 **네 가지 과제**를 시켰다. **네 과제 전부 렌즈에 `Spanish`가 읽혔다.**

`Spanish` → `French`로 바꿔치기하니:

| 과제 | 결과 |
|---|---|
| 보고 / 유연 추론 | 거의 매번 답이 바뀜 ✅ |
| 이어쓰기 / 이상탐지 | **아무 변화 없음** ❌ |

> *"A lens that surfaces a concept in its readout has not necessarily found the direction the model actually computes with."*

**읽힌다 ≠ 인과적으로 쓰인다.** 우리가 8월에 겪은 그 일(bg-focused DIM — 구분은 되는데 건드려도 안 바뀜)과 정확히 같다.

## 1.4 구조적 발견

- **workspace 밴드**: 층을 0~100으로 정규화하면 **L38~L92**. 앞 1/3은 *"noisy and largely uninterpretable"*
- **용량 ~25**: 동시에 올라가는 개념 수. 단일 층에서는 **1~2개**
- **ignition**: 두 임베딩을 섞으면 초반 층은 비율대로 가다가 L38 근처부터 **한쪽으로 붙고 임계에서 급전환** (all-or-none)

## 1.5 리뷰 — 우리 입장에서

**좋은 점**
- 개입(swap/ablate/clamp)으로 기능 주장을 전부 뒷받침했다. 상관만 보고 끝내지 않음
- 렌즈가 **설계상 인과적**이다(출력에 대한 선형화된 인과 효과). tuned lens는 목적함수가 상관 기반
- 코드·평가셋 공개

**주의할 점**
- 저자 스스로 *"The Jacobian lens is an imperfect tool… only approximately and incompletely captures the model's underlying workspace structure"*
- **단일 토큰 개념만** 읽을 수 있다 (토크나이저에 한 토큰으로 있는 단어)
- 개념이 **어떻게 결합돼 있는지는 모른다** — "spider, legs, eight가 있다"까지만
- 각주에서 **top-κ SAE feature가 실제 workspace 방향을 더 잘 근사할 수도 있다**고 인정
- 모델 파라미터 수 비공개 (Claude 계열)


---
# 2. J-lens 원리

## 2.1 문제

모델은 답을 한 번에 안 만든다. **층을 지나며 조금씩** 만든다(27B는 64층).
알고 싶은 것: **"12층에서 모델은 이미 무슨 답을 정했나?"**

끝에 달린 `unembedding`(`W_U`)에 중간 층 벡터를 그냥 넣으면 될 것 같지만 — **안 된다.**
`W_U`는 **마지막 층 기준**으로 학습됐는데 층마다 기준축이 틀어져 있다.

<svg viewBox="0 0 760 240" width="100%" role="img" aria-label="logit lens와 J-lens 비교">
  <text x="120" y="22" font-size="13" fill="#b91c1c" text-anchor="middle">3층에서 그냥 읽으면</text>
  <circle cx="120" cy="120" r="66" fill="none" stroke="#d0d0d0" stroke-width="1.5"/>
  <g transform="rotate(115 120 120)">
    <text x="120" y="66"  font-size="12" fill="#666" text-anchor="middle">정답</text>
    <text x="120" y="182" font-size="12" fill="#aaa" text-anchor="middle">오답</text>
    <text x="62"  y="124" font-size="12" fill="#aaa" text-anchor="middle">기타</text>
    <text x="178" y="124" font-size="12" fill="#aaa" text-anchor="middle">기타</text>
  </g>
  <line x1="120" y1="120" x2="120" y2="62" stroke="#333" stroke-width="3" stroke-linecap="round"/>
  <circle cx="120" cy="120" r="5" fill="#333"/>
  <text x="120" y="212" font-size="12" fill="#b91c1c" text-anchor="middle">눈금판이 돌아가 있다 → 엉뚱한 글자</text>

  <text x="380" y="110" font-size="13" fill="#2563eb" text-anchor="middle">환산표 J₃</text>
  <text x="380" y="130" font-size="12" fill="#2563eb" text-anchor="middle">눈금판을 되돌린다</text>
  <path d="M330 150 L430 150" stroke="#999" stroke-width="2" marker-end="url(#a)"/>
  <defs><marker id="a" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto">
    <path d="M0,0 L6,3 L0,6 z" fill="#999"/></marker></defs>

  <text x="640" y="22" font-size="13" fill="#15803d" text-anchor="middle">되돌리고 읽으면</text>
  <circle cx="640" cy="120" r="66" fill="none" stroke="#15803d" stroke-width="1.5"/>
  <text x="640" y="66"  font-size="12" fill="#15803d" text-anchor="middle">정답</text>
  <text x="640" y="182" font-size="12" fill="#aaa" text-anchor="middle">오답</text>
  <text x="582" y="124" font-size="12" fill="#aaa" text-anchor="middle">기타</text>
  <text x="698" y="124" font-size="12" fill="#aaa" text-anchor="middle">기타</text>
  <line x1="640" y1="120" x2="640" y2="62" stroke="#15803d" stroke-width="3" stroke-linecap="round"/>
  <circle cx="640" cy="120" r="5" fill="#15803d"/>
  <text x="640" y="212" font-size="12" fill="#15803d" text-anchor="middle">바늘은 그대로 → 정답이 읽힌다</text>
</svg>

**바늘** = 모델이 든 정보(`h`) · **눈금판** = 그걸 단어로 읽는 기준(`W_U`)
3층부터 이미 정답을 가리키고 있었다. **눈금판이 돌아가 있어서 못 읽었을 뿐이다.**

## 2.2 수식

```
lens_l(h) = softmax( W_U · norm( J_l @ h ) )
J_l = E[ ∂h_target / ∂h_l ]
```

차이는 **`J_l @` 하나뿐**이다. `use_jacobian=False`로 끄면 그게 곧 logit lens다.

## 2.3 소스코드에서 확인한 구현 사실

| 항목 | 내용 |
|---|---|
| **target 층** | 논문 기본값은 **penultimate(끝에서 두 번째)**. 마지막 블록을 넣으면 noisy artifact 증가 |
| **norm** | `W_U` 앞에 `norm()` 단계가 있다 |
| **제외 위치** | 앞 **16개 위치는 attention sink**라 평균에서 제외, 마지막 위치도 제외 |
| **비용** | 프롬프트당 forward 1 + backward `ceil(d_model/dim_batch)` |
| **코퍼스** | 논문 기본 1000문장이지만 **"10문장으로도 베이스라인을 이긴다"** |
| **⭐ Qwen3.5** | `hf.py`의 `_LAYOUTS`에 `model.language_model`이 있어 **그대로 지원** |
| **⭐ transport** | `lens.transport(h, layer)`가 `J_l @ h`만 따로 제공 → 기존 residual에 바로 적용 |


---
# 3. 우리 연구와의 연결 / 할 일

## 3.1 어디서 막혔나

| 단계 | 내용 | 상태 |
|---|---|---|
| 1 | 생각 없이 즉답하면 함정에 빠진다 | ✅ |
| 2 | 생각을 시키면 정답으로 돌아온다 | ✅ |
| 3 | **함정 표상을 찾아 억제하면 생각 없이도 정답** | ❌ **막힘** |
| 4 | 다른 함정에도 통한다 | ❌ |

3번을 SAE로 5개월 시도 → **세 데이터셋 전부 실패**. 잡은 후보(27B L15 #81663)는 **모든 문장 마지막 토큰에서만 켜지는 위치 feature**, 구분 AUC **0.472(찍기 이하)**.

**원인: 탐색 범위가 너무 넓었다.** `8만 항목 × 64층 × 여러 위치` → 우연히 그럴듯한 게 반드시 나온다.

## 3.2 J-lens로 여는 길 = 범위 좁히기

9/6 리캡:
> **"다음 단계는 답이 실제로 뒤집히는 지점을 먼저 찾는 causal localization이다 — feature 검색은 그 다음이다."**

**J-lens는 범인을 잡는 도구가 아니라 수색 범위를 좁히는 지도다.**

| 조건 | 보는 것 |
|---|---|
| **hostile** (조건 숨김) | 함정 답을 **몇 층부터** 겨누나 |
| **explicit** (조건 명시) | **몇 층에서 꺾이나** |
| **neutral** (단서 제거) | 기준선 |

두 선이 갈라지는 층 = 다음 단계 집중 탐색 구간.

## 3.3 ⚠️ 논문이 주는 경고

**1.3절 선택성이 그대로 우리 리스크다.** 함정 답이 읽힌다고 그 방향이 인과적으로 함정을 만드는 게 아니다.
→ T1-7이 **"실제로 지우거나 키워서 행동이 바뀌는지, 무작위 대조 필수"**로 끝나는 이유.

## 3.4 T1 체크리스트

| # | 항목 | 이 노트북 |
|---|---|---|
| 1 | 논문에서 계산 방식 확인 | ✅ 1·2절 |
| 2 | `jlens.py` 구현 | ✅ **`src/mindscopex_analysis/jlens.py`** (테스트 17개) |
| 3 | 쉬운 문제로 검증 | **8절** |
| 4 | 함정 3버전 × 층 측정 | **9절** |
| 5 | 궤적 그래프 | **9절** |
| 6 | 기존 DIM과 비교 | **9.3절** |
| 7 | 인과 검증 | 별도 작업 |


---
# 4. 개념 검증 (numpy · CPU 즉시 실행)

8층 가짜 모델을 만들되 **층마다 기준축을 일부러 회전**시킨다(`ROT[l]` = 눈금판이 돌아간 상태).
가짜 문장 200개로 `J_l`을 구하고, logit lens와 비교한다. **GPU 불필요, 몇 초.**


In [ ]:
import numpy as np
rng = np.random.default_rng(0)
D, V, L = 24, 10, 8

def rand_rot(d, rng):
    Q, _ = np.linalg.qr(rng.normal(size=(d, d))); return Q

ROT = [rand_rot(D, rng) for _ in range(L)]          # 층마다 눈금판 회전
W   = [0.90 * rng.normal(size=(D, D)) / np.sqrt(D) for _ in range(L)]
U   = rng.normal(size=(V, D)) / np.sqrt(D)          # W_U 역할

def block(h, l):  return ROT[l] @ h + 0.85 * np.tanh(W[l] @ (ROT[l] @ h))
def unembed(h):   return U @ (h / (np.linalg.norm(h) + 1e-9) * np.sqrt(D))

def run(h0):
    hs, h = [h0], h0
    for l in range(L): h = block(h, l); hs.append(h)
    return hs

def jac_block(h, l):
    z = ROT[l] @ h
    return (np.eye(D) + 0.85 * np.diag(1 - np.tanh(W[l] @ z) ** 2) @ W[l]) @ ROT[l]

def jac_l_to_final(h0, l):
    hs, Jm = run(h0), np.eye(D)
    for k in range(l, L): Jm = jac_block(hs[k], k) @ Jm
    return Jm

CORPUS = [rng.normal(size=D) for _ in range(200)]   # "코퍼스"
Jbar = {l: np.mean([jac_l_to_final(h0, l) for h0 in CORPUS], axis=0) for l in range(L)}
print("J_l 적합 완료:", {l: Jbar[l].shape for l in [0, L - 1]})

In [ ]:
h0 = rng.normal(size=D); hs = run(h0)
true_top = int(np.argmax(unembed(hs[L])))

print(f"모델 최종 출력 토큰 = {true_top}\n")
print(f"{'layer':>5} | {'logit-lens':>10} | {'J-lens':>7} | {'J-lens 정답순위':>14}")
print("-" * 52)
ll_hit = jl_hit = 0
for l in range(L):
    ll, jl = unembed(hs[l]), unembed(Jbar[l] @ hs[l])
    a, b = int(np.argmax(ll)), int(np.argmax(jl))
    rank = int((np.argsort(-jl) == true_top).nonzero()[0][0]) + 1
    ll_hit += a == true_top; jl_hit += b == true_top
    print(f"{l:>5} | {a:>10} | {b:>7} | {rank:>14}")
print("-" * 52)
print(f"정답 일치 층수:  logit-lens {ll_hit}/{L}   J-lens {jl_hit}/{L}")

**J-lens가 똑똑한 게 아니다.** 0층에 이미 답 정보가 있었고 logit lens는 눈금판이 안 맞아 못 읽었을 뿐이다.

> ⚠️ 이 장난감은 층이 거의 선형이라 수송이 거의 정확하다. 실제 모델은 논문도 **앞 1/3은 노이즈**라고 명시한다.


---
# 5. Colab 부트스트랩

**여기부터 GPU가 필요하다.** Colab에서 `런타임 → 런타임 유형 변경 → GPU` 선택 후 실행.
로컬에서 실행하면 자동으로 건너뛴다.


In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/JeonDongJun/mindscopex_analysis"
MARK = Path("src") / "mindscopex_analysis" / "__init__.py"

def find_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / MARK).is_file(): return c
    return None

root = find_root()
if not IN_COLAB:
    if root: os.environ["MINDSCOPEX_ROOT"] = str(root)
    print(f"[local] 부트스트랩 건너뜀. 4절까지는 로컬에서 실행 가능.")
    print(f"        5절부터는 Colab에서 열어야 한다. MINDSCOPEX_ROOT={root}")
else:
    if root is None:
        wd = Path("/content/mindscopex_analysis")
        if (wd / MARK).is_file():
            subprocess.run(["git", "-C", str(wd), "pull", "--ff-only"], check=False)
        else:
            if wd.exists(): shutil.rmtree(wd)
            subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(wd)])
        root = wd
    os.environ["MINDSCOPEX_ROOT"] = str(root.resolve()); os.chdir(root)

    REV = "b70d02fc724d04c916832ca4ead03ff05e8fb1ee"      # Qwen3.5 지원 transformers
    if subprocess.run([sys.executable, "-c", "from transformers import AutoModelForMultimodalLM"],
                      capture_output=True).returncode != 0:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
            f"transformers @ git+https://github.com/huggingface/transformers.git@{REV}",
            "torchvision", "pillow"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/anthropics/jacobian-lens.git"])
    print("[colab] ready:", root)

---
# 6. Qwen 로드 + jlens 연결

**2B로 먼저 검증한다.** 되는 걸 확인한 뒤 27B로 올린다.


In [ ]:
import torch, transformers, jlens

MODEL_ID = "Qwen/Qwen3.5-2B-Base"     # 검증용. 본 실험은 "Qwen/Qwen3.5-27B"

hf  = transformers.AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
tok = transformers.AutoTokenizer.from_pretrained(MODEL_ID)

model = jlens.from_hf(hf, tok)        # 레이아웃 자동 탐지
print(model)
print("탐지된 layout:", model.layout)  # Qwen3.5면 path='model.language_model'
print("n_layers:", model.n_layers, "/ d_model:", model.d_model)

> ✅ **T1 확인 포인트:** `layout`이 `Layout(path='model.language_model', ...)`로 찍히면 Qwen3.5가 별도 어댑터 없이 지원된다는 증거다.


---
# 7. 렌즈 확보

두 경로가 있다. **(a)를 먼저 시도하고 실패하면 (b).**

- **(a) 공개 렌즈** — 논문에 Neuronpedia에 오픈모델용 렌즈가 호스팅돼 있다고 언급됨. 있으면 비용 0
- **(b) 직접 적합** — 논문 기본은 1000문장이지만 **"10문장으로도 베이스라인을 이긴다"**


In [ ]:
# 비용 먼저 확인 (fitting.py 기준: 프롬프트당 backward = ceil(d_model/dim_batch))
for name, d in [("2B", 2048), ("9B", 4096), ("27B", 5120)]:
    for db in (8, 16):
        p = -(-d // db)
        print(f"{name:>4} d_model={d:>5} dim_batch={db:>3} → 프롬프트당 {p:>4}회, 10문장 {p*10:>6,}회, 100문장 {p*100:>7,}회")

In [ ]:
LENS_PATH = "jacobian_lens_2b.pt"

if os.path.exists(LENS_PATH):
    lens = jlens.JacobianLens.load(LENS_PATH)
    print("기존 렌즈 로드:", lens)
else:
    # (b) 직접 적합. 우선 작게: 20문장 × 일부 층
    from datasets import load_dataset          # 없으면: pip install datasets
    ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train", streaming=True)
    prompts, it = [], iter(ds)
    while len(prompts) < 20:
        t = next(it)["text"].strip()
        if len(t.split()) > 60: prompts.append(t)

    # 논문 기본값에 맞춤: target은 penultimate 층
    SCAN = list(range(0, model.n_layers, 2))   # 2층 간격. 전층은 비쌈
    lens = jlens.fit(model, prompts=prompts,
                     source_layers=SCAN, target_layer=-2,
                     dim_batch=16, checkpoint_path="ckpt_2b.pt")
    lens.save(LENS_PATH)
    print("적합 완료:", lens)

---
# 8. 쉬운 문제로 검증 (T1-3)

렌즈가 제대로 붙었는지 확인. **뒤쪽 층에서 정답 단어가 올라오고, logit lens보다 이른 층에서 잡혀야** 한다.


In [ ]:
prompt = "Fact: The currency used in the country shaped like a boot is"

lens_logits, model_logits, ids = lens.apply(model, prompt, positions=[-1])
ll_logits, _, _ = lens.apply(model, prompt, positions=[-1], use_jacobian=False)  # logit lens

print(f"{'layer':>6} | {'J-lens top5':<46} | logit-lens top5")
print("-" * 110)
for layer in sorted(lens_logits):
    j = [tok.decode([t]) for t in lens_logits[layer][0].topk(5).indices]
    l = [tok.decode([t]) for t in ll_logits[layer][0].topk(5).indices]
    print(f"{layer:>6} | {str(j):<46} | {l}")

---
# 9. 함정 문제에 적용 (T1-4·5)

## 9.1 데이터 로드

> 🚧 **블로커:** `canonical_traps_v1`이 저장소에 없다(최신 커밋 8/23). 우선 `goal_affordance_traps_v1`로 진행하고, 확보되면 `DATASET`만 바꾸면 된다.


In [ ]:
ROOT = os.environ.get("MINDSCOPEX_ROOT", ".")
SRC = os.path.join(ROOT, "src")
if SRC not in sys.path: sys.path.insert(0, SRC)
from mindscopex_analysis import lure_dataset_cases

DATASET = "goal_affordance_traps_v1"      # canonical_traps_v1 확보되면 교체
cases = lure_dataset_cases(DATASET)

by_pair = {}
for c in cases: by_pair.setdefault(c.pair_id, {})[c.condition] = c

PAIR = "target_transport_car_wash"        # 세차장
for cond in ("hostile", "explicit", "neutral"):
    print(f"[{cond}]\n  {by_pair[PAIR][cond].prompt[:150]}...\n")
print("정답:", by_pair[PAIR]["hostile"].correct_answer,
      "| 함정:", by_pair[PAIR]["hostile"].lure_answer)

## 9.2 층별 궤적 측정

각 조건에서 **함정 답의 첫 토큰이 몇 위인지**를 층마다 추적한다. 순위가 낮을수록(1위에 가까울수록) 강하게 겨누는 중.


In [ ]:
from mindscopex_analysis.jlens import (
    condition_trajectories, divergence_layer, trajectory_rows, answers_are_separable,
)

# 렌즈는 단일 토큰만 읽는다 → 두 답의 첫 토큰이 다른지 먼저 확인
case = by_pair[PAIR]["hostile"]
assert answers_are_separable(tok, case), (
    f"첫 토큰이 같아 구분 불가: {case.correct_answer!r} vs {case.lure_answer!r}"
)

traj = condition_trajectories(lens, model, tok, by_pair[PAIR])
print("측정 완료:", {c: len(traj[c]) for c in traj}, "개 층")

layer, ratio = divergence_layer(traj, base="hostile", control="explicit")
print(f"\nhostile과 explicit이 가장 갈라지는 층: L{layer} (비 {ratio:.1f}배)")
print("→ 이후 SAE feature 탐색은 이 구간에 집중한다 (causal localization)")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
for cond, style in [("hostile", "-o"), ("explicit", "--s"), ("neutral", ":^")]:
    if cond not in traj: continue
    ls = sorted(traj[cond])
    ax.plot(ls, [traj[cond][l].lure_rank for l in ls], style, label=cond, ms=4)
ax.axvline(layer, color="crimson", lw=1, ls="--", alpha=.6)
ax.annotate(f"L{layer}", (layer, 1), color="crimson", fontsize=9,
            xytext=(4, 4), textcoords="offset points")
ax.set_yscale("log"); ax.invert_yaxis()
ax.set_xlabel("layer"); ax.set_ylabel("함정 답 순위 (낮을수록 강하게 겨눔)")
ax.set_title(f"{PAIR} — 조건별 함정 답 궤적")
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

# CSV로 저장해 팀과 공유
import csv, pathlib
rows = trajectory_rows(traj)
out = pathlib.Path("outputs") / f"jlens_traj_{PAIR}.csv"
out.parent.mkdir(exist_ok=True)
with out.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
print("저장:", out)

## 9.3 기존 DIM 방향과 비교 (T1-6)

`lens.transport()`가 `J_l @ h`만 따로 제공하므로, 우리가 이미 뽑아둔 residual에 그대로 적용된다.


In [ ]:
from mindscopex_analysis.jlens import lens_answer_direction, first_token_id

# 9.2에서 찾은 층에서 "함정 - 정답" 방향을 뽑는다.
# W_U의 행이 토큰 방향 → J_l로 밀어넣으면 그 층에서의 방향이 된다.
d = lens_answer_direction(
    lens, model._lm_head.weight, layer=layer,
    lure_token_id=first_token_id(tok, case.lure_answer),
    correct_token_id=first_token_id(tok, case.correct_answer),
)
print(f"L{layer} 함정-정답 방향 norm: {d.norm():.3f}  shape: {tuple(d.shape)}")
print("→ 기존 DIM 산출물과 F.cosine_similarity로 비교 (T1-6)")
print("⚠️ 이건 '읽기' 방향이다. 건드려서 행동이 바뀌는지는 T1-7에서 별도 검증")

---
# 10. 한계 · 진행 상황

## 논문이 밝힌 한계 (직접 인용)

- *"The Jacobian lens is an imperfect tool, which we believe only approximately and incompletely captures the model's underlying workspace structure."*
- **단일 토큰 개념만** — 토크나이저에 한 토큰으로 있는 단어만 읽힌다
- **결합 구조는 모른다** — "spider, legs, eight가 있다"까지만, 어떻게 묶였는지는 모름
- *"in roughly the first third of the model, the readouts are noisy and largely uninterpretable"*
- 각주에서 **top-κ SAE feature가 실제 workspace 방향을 더 잘 근사할 수도 있다**고 인정

## T1 진행 상황

| # | 항목 | 상태 |
|---|---|---|
| 1 | 논문 계산 방식 확인 | ✅ 논문 본문 + 소스 전체 |
| 2 | `jlens.py` 구현 | ✅ **완료** — `src/mindscopex_analysis/jlens.py`, 단위테스트 17개 통과 |
| 3 | 쉬운 문제 검증 | 🟡 코드 완성, **GPU 실행 대기** |
| 4·5 | 함정 궤적 | 🟡 코드 완성, **데이터셋 대기** |
| 6 | DIM 비교 | 🟡 경로 확인, DIM 산출물 필요 |
| 7 | 인과 검증 | ⬜ 별도 작업 |

## 🚧 블로커 · 논의 필요

1. **`canonical_traps_v1`이 저장소에 없다** (최신 커밋 8/23) — 공유 필요
2. **T1-2 자체 구현 여부** — 공식 패키지가 Qwen3.5를 지원하므로 건너뛰어도 될지
3. **Qwen용 공개 렌즈 존재 여부** — 있으면 적합 비용 0
4. **DIM 산출물 위치** — 9.3절 비교에 필요
